In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
import scipy.stats as stats

import glob

In [9]:
files = glob.glob("../data/raw/*_historical.csv")

print(files)

[]


In [7]:


df_list = []

for file in files:
    df = pd.read_csv(file)
    df["time"] = pd.to_datetime(df["time"])
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)

df.head()

['.gitkeep']


ValueError: No objects to concatenate

In [ ]:
summary = df.groupby("city").agg(
    ["count", "mean", "std", "min", "median", "max"]
)

summary

skew_kurt = df.groupby("city").agg(["skew", "kurt"])
skew_kurt

df["year"] = df["time"].dt.year

yearly = df.groupby(["city", "year"]).agg({
    "temperature_2m_max": "mean",
    "precipitation_sum": "sum",
    "windspeed_10m_max": "max"
})

yearly

df["month"] = df["time"].dt.month

monthly = df.groupby(["city", "month"]).agg(
    ["mean", "std"]
)

monthly



In [ ]:
sns.histplot(data=df, x="temperature_2m_max", hue="city", kde=True)
plt.title("Temperature Distribution by City")
plt.show()

df["season"] = df["month"] % 12 // 3 + 1

sns.boxplot(data=df, x="season", y="temperature_2m_max", hue="city")
plt.title("Temperature by Season")
plt.show()

sns.violinplot(data=df, x="city", y="precipitation_sum")
plt.yscale("log")
plt.title("Rainfall Distribution")
plt.show()

city_df = df[df["city"] == "Baku"]

stats.probplot(city_df["temperature_2m_max"], dist="norm", plot=plt)
plt.title("QQ Plot - Baku Temperature")
plt.show()



In [ ]:
for city in df["city"].unique():
    city_df = df[df["city"] == city].sort_values("time")

    plt.figure(figsize=(12,4))
    plt.plot(city_df["time"], city_df["temperature_2m_max"], alpha=0.5)
    plt.plot(city_df["time"], city_df["temperature_2m_max"].rolling(30).mean(), color="red")
    
    plt.title(f"{city} Temperature Trend")
    plt.show()

baku = df[df["city"] == "Baku"].set_index("time")

decomp = seasonal_decompose(baku["temperature_2m_max"], model="additive", period=365)
decomp.plot()
plt.show()

baku["day_of_year"] = baku.index.dayofyear

sns.lineplot(data=baku, x="day_of_year", y="temperature_2m_max", hue=baku.index.year)
plt.title("Year-over-Year Temperature (Baku)")
plt.show()

pivot = baku.pivot_table(
    values="temperature_2m_max",
    index=baku.index.dayofyear,
    columns=baku.index.year
)

sns.heatmap(pivot, cmap="coolwarm")
plt.title("Temperature Heatmap (Baku)")
plt.show()



In [ ]:
df_2023 = df[df["year"] == 2023]

sns.lineplot(data=df_2023, x="time", y="temperature_2m_max", hue="city")
plt.title("City Comparison (2023)")
plt.show()

sample_city = df[df["city"] == "Baku"]

sns.pairplot(sample_city[[
    "temperature_2m_max",
    "precipitation_sum",
    "windspeed_10m_max",
    "relative_humidity_2m_mean"
]])

corr = sample_city.select_dtypes(include=np.number).corr()

sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix (Baku)")
plt.show()



### Key Findings

1. Temperature shows strong seasonal patterns across all cities.
2. Antalya has more stable and warmer temperatures compared to others.
3. Precipitation is highly skewed, with many zero-rain days.
4. Some years appear warmer than others, suggesting possible trends.
5. Humidity and temperature show moderate correlation.

### Questions for Hypothesis Testing

1. Is average temperature significantly different between cities?
2. Does precipitation differ significantly across seasons?
3. Is there a statistically significant warming trend over years?

### Data Quality Issues

- Some variables contain occasional missing values.
- Precipitation distribution is highly skewed.
- No major gaps in time series detected.